<a href="https://colab.research.google.com/github/tapiaj-git/DS201/blob/main/practice_paths_and_repos_John_Tapia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paths, Files, and Directories in Google Colab

## Learning objective

The main goal of this notebook is to understand how to **tell your code where a file is located**.

By the end, you should be able to:

- identify your current working directory;
- distinguish between **absolute** and **relative** paths;
- understand how directories are organized;
- use Python to search for files and count files;
- use a path to read a CSV file with `pandas`.

We will use a few Linux commands when they are convenient, but the main data tasks will be done in Python.

## 1. Colab is a remote Linux machine

When you run a Google Colab notebook, your code is usually running on a temporary remote Linux machine provided by Google.

It is **not running directly on your laptop**.

The Colab runtime has its own directories and files. Its default working directory is usually:

```text
/content
```

A few Linux commands are useful for quickly seeing where we are and what is around us.

In [5]:
!pwd

/content


`pwd` means **print working directory**.

Now list the contents of the current directory:

In [6]:
!ls

sample_data


A slightly more detailed listing is:

In [7]:
!ls -la

total 16
drwxr-xr-x 1 root root 4096 Aug 24 13:21 .
drwxr-xr-x 1 root root 4096 Sep  2 01:24 ..
drwxr-xr-x 4 root root 4096 Aug 24 13:21 .config
drwxr-xr-x 1 root root 4096 Aug 24 13:21 sample_data


These commands are useful for orientation:

- `pwd` → where am I?
- `ls` → what is here?
- `ls -la` → what is here, including hidden files?

For the rest of the notebook, we will mostly use Python to work with paths and files.

## 2. Absolute and relative paths

A **path** tells the computer where a file or directory is located.

### Absolute path

An absolute path gives the complete location starting from the root `/`.

Example:

```text
/content/project/data/houses.csv
```

### Relative path

A relative path describes the location starting from the **current working directory**.

If the current working directory is:

```text
/content
```

then the same file could be written as:

```text
project/data/houses.csv
```

The same file can therefore have both an absolute path and a relative path.

Python's `pathlib` library makes paths easy to work with.

In [8]:
from pathlib import Path

current_dir = Path.cwd()

print("Current directory:", current_dir)
print("Is this absolute?", current_dir.is_absolute())

Current directory: /content
Is this absolute? True


## 3. A small example

Let's create a very small directory structure:

```text
path_demo/
    data/
        sample.csv
```

In [9]:
project_dir = Path("/content/path_demo")
data_dir = project_dir / "data"

data_dir.mkdir(parents=True, exist_ok=True)

sample_file = data_dir / "sample.csv"
sample_file.write_text(
    "city,price\n"
    "Boston,500000\n"
    "Atlanta,350000\n"
)

print(sample_file)

/content/path_demo/data/sample.csv


The `/` operator joins pieces of a path.

For example:

```text
project directory
    ↓
data directory
    ↓
sample.csv
```

Now compare an absolute and relative path to the same file.

In [10]:
absolute_path = Path("/content/path_demo/data/sample.csv")
relative_path = Path("path_demo/data/sample.csv")

print("Absolute path:", absolute_path)
print("Relative path:", relative_path)

print("\nAbsolute path exists:", absolute_path.exists())
print("Relative path exists:", relative_path.exists())

Absolute path: /content/path_demo/data/sample.csv
Relative path: path_demo/data/sample.csv

Absolute path exists: True
Relative path exists: True


Because the current working directory is `/content`, the relative path:

```text
path_demo/data/sample.csv
```

points to the same file as:

```text
/content/path_demo/data/sample.csv
```

Python can convert the relative path into an absolute path:

In [ ]:
relative_path.resolve()

## 4. Read a file using its path

Once you know the correct path, you can give it directly to `pandas`.

In [17]:
import pandas as pd

df = pd.read_csv(relative_path)
df

,city,price
0,Boston,500000
1,Atlanta,350000


This is the central idea for the notebook:

> A data-reading function can only open the correct file if you give it the correct path.

# Activity: Explore a COVID-19 Tweets Dataset

We will work with:

```text
https://github.com/lopezbec/COVID19_Tweets_Dataset_2022
```

This repository contains the 2022 data.

This might take several minutes since the repo is very large

## 5. Clone the repository

Here it is appropriate to use a shell command because `git clone` is a standard Git command (this might take a few minuted ~10-15min).

Run:

In [18]:
!time git clone https://github.com/lopezbec/COVID19_Tweets_Dataset_2022

Cloning into 'COVID19_Tweets_Dataset_2022'...
remote: Enumerating objects: 5859, done.
remote: Total 5859 (delta 0), reused 0 (delta 0), pack-reused 5859 (from 1)
Receiving objects: 100% (5859/5859), 4.45 GiB | 15.41 MiB/s, done.
Resolving deltas: 100% (1716/1716), done.
Updating files: 100% (5836/5836), done.

real	9m53.036s
user	9m12.611s
sys	1m13.533s


Confirm that the repository directory now exists:

In [19]:
!pwd
!ls

/content
COVID19_Tweets_Dataset_2022  path_demo	sample_data


The repository now has an absolute path:

```text
/content/COVID-19-TweetIDs
```

From `/content`, its relative path is:

```text
COVID-19-TweetIDs
```

We will now switch back to Python for searching and counting files.

## 6. Get the repository's file paths into Python

The repository has already been cloned into the Colab runtime, so all of its files are available in the filesystem.

We will now use Python to search through the repository and create a list called `repo_files`.

Each item in `repo_files` will be a `Path` object representing a **relative path from the repository root**.

This will let us use Python to:

- inspect the directory structure;
- search for specific files;
- count files;
- build paths to files we want to read.

In [23]:
from pathlib import Path

repo_dir = Path("COVID19_Tweets_Dataset_2022")

repo_files = [
    file for file in repo_dir.rglob("*")
    if file.is_file() and ".git" not in file.parts
]

print("Number of files:", len(repo_files))

Number of files: 5836


Inspect a few paths:

In [24]:
repo_files[:10]

[PosixPath('COVID19_Tweets_Dataset_2022/Get_twitter_Summary_Files.ipynb'),
 PosixPath('COVID19_Tweets_Dataset_2022/Features_table.csv'),
 PosixPath('COVID19_Tweets_Dataset_2022/Automatically_Hydrate_TweetsIDs_COVID19_v3.ipynb'),
 PosixPath('COVID19_Tweets_Dataset_2022/README.md'),
 PosixPath('COVID19_Tweets_Dataset_2022/Summary_Details/2022_06/2022_06_06_01_Summary_Details.csv'),
 PosixPath('COVID19_Tweets_Dataset_2022/Summary_Details/2022_06/2022_06_03_17_Summary_Details.csv'),
 PosixPath('COVID19_Tweets_Dataset_2022/Summary_Details/2022_06/2022_06_25_23_Summary_Details.csv'),
 PosixPath('COVID19_Tweets_Dataset_2022/Summary_Details/2022_06/2022_06_12_14_Summary_Details.csv'),
 PosixPath('COVID19_Tweets_Dataset_2022/Summary_Details/2022_06/2022_06_16_10_Summary_Details.csv'),
 PosixPath('COVID19_Tweets_Dataset_2022/Summary_Details/2022_06/2022_06_26_17_Summary_Details.csv')]

Because these are Python `Path` objects, you can inspect information such as:

- `.name` → filename;
- `.suffix` → extension;
- `.parts` → components of the path;
- `.parent` → directory containing the file.

In [25]:
example_path = repo_files[0]

print("Path:", example_path)
print("Name:", example_path.name)
print("Suffix:", example_path.suffix)
print("Parts:", example_path.parts)
print("Parent:", example_path.parent)

Path: COVID19_Tweets_Dataset_2022/Get_twitter_Summary_Files.ipynb
Name: Get_twitter_Summary_Files.ipynb
Suffix: .ipynb
Parts: ('COVID19_Tweets_Dataset_2022', 'Get_twitter_Summary_Files.ipynb')
Parent: COVID19_Tweets_Dataset_2022


## 7. Activity A — Count files with Python

Use Python to determine how many files are located in each **top-level directory** of the repository.

### Hints

Think about:

- each item in `repo_files` is a relative path;
- each path is made of multiple components;
- the first component tells you which top-level directory the file belongs to;
- you need to count how many files belong to each top-level directory.

You may use a dictionary, `Counter`, or another Python approach.

Do not count the files manually.

In [33]:
# Your Python code here.
counts = {}

for path in repo_files:
    folder = path.parts[1]
    counts[folder] = counts.get(folder, 0) + 1

print(counts)

{'Get_twitter_Summary_Files.ipynb': 1, 'Features_table.csv': 1, 'Automatically_Hydrate_TweetsIDs_COVID19_v3.ipynb': 1, 'README.md': 1, 'Summary_Details': 5832}


## 8. Activity B — Search for files with Python

Use `repo_files` to find the file corresponding to **Valentine's Day 2020 at noon**.

For this activity, look for:

- date: **February 14, 2020**
- hour: **12:00**

### Hints

Think about:

- how the date appears in the directory and file names;
- how the hour is represented in the filename;
- how you can inspect the `.name` or `.parts` of each `Path`;
- whether searching for parts of the date and hour can help you narrow down the results.

Try to use Python to filter `repo_files` until you identify the correct file.

Your goal is to practice using the information encoded in a path to locate a specific file.

In [68]:
# Your Python code here.
print(repo_files[400])

for path in repo_files:
    if "2022_02_14_12" in path.name:
        print(path)

COVID19_Tweets_Dataset_2022/Summary_Details/2022_06/2022_06_23_13_Summary_Details.csv
COVID19_Tweets_Dataset_2022/Summary_Details/2022_02/2022_02_14_12_Summary_Details.csv


## 9. Activity C — Count the Tweets

Now that you have identified the file corresponding to **Valentine's Day 2020**, use Python to determine how many tweets were collected during day.

### Your task

1. Use the path you found in the previous activity.
2. Read the file into Python.
3. Inspect the data to understand how the tweet IDs are stored.
4. Count how many tweets are contained in the files.
5. Report your result.

### Hints

Think about:

- what type of file you are working with;
- which Python function can be used to open and read that type of file;
- whether each line represents one tweet ID;
- whether you need to count rows, lines, or unique values;
- how the path you found in the previous activity tells Python exactly which file to read.

Before reporting your answer, inspect a few records to make sure you loaded the file you intended.

### Final question

> **How many tweets were collected on February 14, 2020?**



In [78]:
from functools import total_ordering
# Calculate and report your answer here.
import pandas as pd

val_files = []

for path in repo_files:
    if "2022_02_14" in path.name:
        val_files.append(path)


total = 0
for path in val_files:
  df = pd.read_csv(path)
  total += len(df)

print(total)

2443201


# There seems to have been a total of 2,443,201 tweets on Feb 14th, 2022. Used 2022 since 2020 data DNE. I got this by appending a empty list with all the paths in repo that was dated for Feb 14th. Then I did another loop where I turned it into a daatframe using pandas and got the total count for each one. I just kept adding it on to the total variable that started at 0.


# 1203,30303


## Using GenAI

You may use GenAI to help generate or debug your code.

However, you should be able to explain:

- what your current working directory is;
- whether a path is relative or absolute;
- how your Python code searched for the correct file;
- why the selected path corresponds to the requested date and hour;
- how the path was passed to `pandas`.

If GenAI suggests a path, verify it against the repository rather than assuming it is correct.